
# Buy Fee Model Dataset Builder v6

## Objective

Create a **clean, defensible event-level dataset** for later Buy Fee modeling.

This notebook does **not** train a model. Its purpose is to make sure the data going into the future model is correct.

## Locked business rules

### Sold logic
A unit is sold only when:

- `VRSTATUS == "HST"` and `VRINVOICE != "RETURN"`
- OR
- `VRSTATUS == "SLD"` and `VRIFORTRY != "Y"`

### Unsold inventory
- `NOS` = valid unsold/offered inventory

### Exclusions
- `INV` = excluded
- `SAL` = excluded

### Price tier
- **`SALEAMT_FINAL` is used for both sold and unsold units**
- Default tier width = **$1,000**

### Buy Fee / RPU
- Use **`BUYFEE_FINAL` only**
- Do **not** use `Buyer_BuyFee`
- Buy Fee revenue is counted only on sold units
- RPU = Buy Fee Revenue / Sold Units

### Fee-change event
An actual treatment event exists only when:

`BUYFEE_CHANGE_STATUS == "Change"`

Events are deduplicated at:

**Location × Lane × Price Tier × Change Date**

### Primary 8-week quality rules
An event is eligible only when:

- Pre registered units >= **25**
- Post registered units >= **25**
- Pre observed weeks >= **6 of 8**
- Post observed weeks >= **6 of 8**

RPU eligibility additionally requires sufficient sold-unit and fee coverage.

### Diagnostics
The notebook explicitly reports:

- Unique locations
- Status counts
- Sold vs unsold counts
- Fee-change event counts
- Corporate change dates
- Location × change-date counts
- 8-week coverage
- Threshold sensitivity
- RPU coverage using `BUYFEE_FINAL`


In [1]:

from pathlib import Path
import pandas as pd
import numpy as np
import gc
import warnings

warnings.filterwarnings("ignore")

# ============================================================
# USER CONFIGURATION
# ============================================================

DATA_FOLDER = Path(
    r"C:\Users\ViditSheth\OneDrive - America's Auto Auction\Finance - Documents\FP&A\Data Analytics\Development\Vidit_Sanbox\Buy Fee Model Phase 2\Rough Area\Volume Attirtion"
)

INPUT_CSV = DATA_FOLDER / "VIN_LEVEL_DATA_WITH_DATES_AND_SALE_PRICE_NEW_DATA.csv"

OUTPUT_DIR = DATA_FOLDER / "buy_fee_model_dataset_v6"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

CHUNK_SIZE = 250_000
ENCODING = "cp1252"

PRICE_TIER_WIDTH = 1000
WINDOWS = [4, 8, 12]
PRIMARY_WINDOW = 8

# Primary event quality rules
MIN_REGISTERED_PRE = 25
MIN_REGISTERED_POST = 25
MIN_OBSERVED_WEEKS_PRE = 6
MIN_OBSERVED_WEEKS_POST = 6

# RPU quality rules
MIN_SOLD_PRE_RPU = 15
MIN_SOLD_POST_RPU = 15
MIN_BUYFEE_COVERAGE = 0.90

# Overlap flagging
OVERLAP_LOOKBACK_WEEKS = 8
OVERLAP_LOOKAHEAD_WEEKS = 8

print("=" * 80)
print("V6 CONFIGURATION")
print("=" * 80)
print("Input CSV :", INPUT_CSV)
print("Exists    :", INPUT_CSV.exists())
print("Output dir:", OUTPUT_DIR)

if not INPUT_CSV.exists():
    print("\nCSV files found:")
    for f in DATA_FOLDER.glob("*.csv"):
        print(" -", f.name)
    raise FileNotFoundError(INPUT_CSV)


V6 CONFIGURATION
Input CSV : C:\Users\ViditSheth\OneDrive - America's Auto Auction\Finance - Documents\FP&A\Data Analytics\Development\Vidit_Sanbox\Buy Fee Model Phase 2\Rough Area\Volume Attirtion\VIN_LEVEL_DATA_WITH_DATES_AND_SALE_PRICE_NEW_DATA.csv
Exists    : True
Output dir: C:\Users\ViditSheth\OneDrive - America's Auto Auction\Finance - Documents\FP&A\Data Analytics\Development\Vidit_Sanbox\Buy Fee Model Phase 2\Rough Area\Volume Attirtion\buy_fee_model_dataset_v6


## 1. Detect required columns

In [2]:

header_df = pd.read_csv(
    INPUT_CSV,
    nrows=10,
    encoding=ENCODING,
    encoding_errors="replace",
    low_memory=False
)

ALIASES = {
    "vin": ["VRVIN", "VIN"],
    "status": ["VRSTATUS"],
    "sale_date": ["VRSALEDATE", "SALEDATE"],
    "saleamt_final": ["SALEAMT_FINAL"],
    "location": ["VRLOCATION", "LOCATION", "Location"],
    "lane": ["LANE", "Lane", "ACTUAL_LANE"],
    "invoice": ["VRINVOICE"],
    "ifortry": ["VRIFORTRY"],
    "buyfee_final": ["BUYFEE_FINAL", "BUYFEE-FINAL", "BUYFEE FINAL"],
    "change_status": ["BUYFEE_CHANGE_STATUS"],
    "change_date": ["BUYFEE_CHANGE_DATE"],
    "previous_buyfee": ["PREVIOUS_BUYFEE"],
    "change_amount": ["BUY_FEE_CHANGE_AMOUNT"],
    "vehicle_year": ["Vehicle_year", "VEHICLE_YEAR"],
    "mileage": ["VRMILEAGE", "MILEAGE"],
    "make": ["Make", "MAKE"],
    "model": ["Model", "MODEL"],
    "condition1": ["Vehicle_condition_grade_1", "CONDITION_GRADE_1"],
    "condition2": ["Vehicle_condition_grade_2", "CONDITION_GRADE_2"],
    "condition_overall": ["Vehicle_condition_overall", "CONDITION_OVERALL"],
    "sales_channel": ["Misc_SalesChannel", "SALES_CHANNEL"],
    "high_bid": ["VRHIGHBID", "HIGH_BID"],
    "floor": ["VRFLOOR", "FLOOR"],
}

def resolve_col(columns, aliases):
    lookup = {str(c).strip().upper(): c for c in columns}
    for a in aliases:
        key = a.strip().upper()
        if key in lookup:
            return lookup[key]
    return None

COL = {k: resolve_col(header_df.columns, v) for k, v in ALIASES.items()}

mapping = pd.DataFrame(
    [{"Logical_Field": k, "Resolved_Column": v} for k, v in COL.items()]
)
display(mapping)

required = [
    "status",
    "sale_date",
    "saleamt_final",
    "location",
    "lane",
    "invoice",
    "ifortry",
    "buyfee_final",
    "change_status",
    "change_date",
    "previous_buyfee",
    "change_amount"
]

missing = [k for k in required if COL[k] is None]

if missing:
    raise ValueError(
        "Missing required columns: "
        + ", ".join(missing)
        + "\nPlease verify the source column names before continuing."
    )


,Logical_Field,Resolved_Column
0,vin,VRVIN
1,status,VRSTATUS
2,sale_date,VRSALEDATE
3,saleamt_final,SALEAMT_FINAL
4,location,VRLOCATION
5,lane,LANE
6,invoice,VRINVOICE
7,ifortry,VRIFORTRY
8,buyfee_final,BUYFEE-FINAL
9,change_status,BUYFEE_CHANGE_STATUS


## 2. Helper functions

In [3]:

def clean_text(s):
    return s.astype("string").str.strip()

def to_num(s):
    return pd.to_numeric(s, errors="coerce")

def parse_date_series(s):
    x = s.astype("string").str.strip()
    x2 = x.str.replace(r"\.0$", "", regex=True)

    out = pd.Series(pd.NaT, index=s.index, dtype="datetime64[ns]")

    mask8 = x2.str.fullmatch(r"\d{8}", na=False)

    if mask8.any():
        out.loc[mask8] = pd.to_datetime(
            x2.loc[mask8],
            format="%Y%m%d",
            errors="coerce"
        )

    if (~mask8).any():
        out.loc[~mask8] = pd.to_datetime(
            x.loc[~mask8],
            errors="coerce"
        )

    return out

def exact_sold_flag(df):
    """
    Locked AAA sold logic:
    (VRSTATUS == HST and VRINVOICE != RETURN)
    OR
    (VRSTATUS == SLD and VRIFORTRY != Y)
    """
    status = clean_text(df[COL["status"]]).str.upper()
    invoice = clean_text(df[COL["invoice"]]).str.upper()
    ifortry = clean_text(df[COL["ifortry"]]).str.upper()

    sold = (
        (status.eq("HST") & ~invoice.eq("RETURN"))
        |
        (status.eq("SLD") & ~ifortry.eq("Y"))
    )

    return sold.fillna(False).astype(bool)

def valid_inventory_flag(df):
    """
    Valid standard-model inventory:
    HST, SLD, NOS only.
    INV and SAL are explicitly excluded.
    """
    status = clean_text(df[COL["status"]]).str.upper()
    return status.isin(["HST", "SLD", "NOS"]).fillna(False)

def make_price_tier(saleamt_final):
    amt = pd.to_numeric(saleamt_final, errors="coerce")

    low = np.floor(amt / PRICE_TIER_WIDTH) * PRICE_TIER_WIDTH
    high = low + PRICE_TIER_WIDTH

    label = pd.Series(pd.NA, index=amt.index, dtype="string")

    valid = amt.notna() & (amt >= 0)

    label.loc[valid] = (
        low.loc[valid].astype("Int64").astype(str)
        + "-"
        + high.loc[valid].astype("Int64").astype(str)
    )

    return low, high, label

def safe_pct_change(post, pre):
    if pd.isna(pre) or pre == 0 or pd.isna(post):
        return np.nan
    return (post - pre) / pre

def linear_slope(values):
    vals = np.asarray(values, dtype=float)
    mask = np.isfinite(vals)

    if mask.sum() < 2:
        return np.nan

    x = np.arange(len(vals))[mask]
    y = vals[mask]

    return np.polyfit(x, y, 1)[0]

def coefficient_of_variation(values):
    vals = np.asarray(values, dtype=float)
    vals = vals[np.isfinite(vals)]

    if len(vals) < 2:
        return np.nan

    mean = vals.mean()

    if mean == 0:
        return np.nan

    return vals.std(ddof=1) / abs(mean)


## 3. Pass 1 — create robust weekly panel and fee-change events

In [4]:

weekly_parts = []
event_parts = []

rows_processed = 0
valid_inventory_rows = 0
sold_rows = 0
nos_rows = 0
excluded_inv_rows = 0
excluded_sal_rows = 0
sold_with_buyfee = 0

status_counts = {}
unique_locations = set()
unique_lanes = set()

reader = pd.read_csv(
    INPUT_CSV,
    chunksize=CHUNK_SIZE,
    encoding=ENCODING,
    encoding_errors="replace",
    low_memory=False
)

for chunk_num, chunk in enumerate(reader, start=1):
    rows_processed += len(chunk)

    status = clean_text(chunk[COL["status"]]).str.upper()

    counts = status.value_counts(dropna=False)
    for k, v in counts.items():
        status_counts[str(k)] = status_counts.get(str(k), 0) + int(v)

    sold = exact_sold_flag(chunk)
    valid_inventory = valid_inventory_flag(chunk)

    valid_inventory_rows += int(valid_inventory.sum())
    sold_rows += int(sold.sum())
    nos_rows += int(status.eq("NOS").sum())
    excluded_inv_rows += int(status.eq("INV").sum())
    excluded_sal_rows += int(status.eq("SAL").sum())

    sale_dt = parse_date_series(chunk[COL["sale_date"]])
    saleamt_final = to_num(chunk[COL["saleamt_final"]])

    loc = clean_text(chunk[COL["location"]]).str.upper()
    lane = clean_text(chunk[COL["lane"]]).str.upper()

    buyfee_final = to_num(chunk[COL["buyfee_final"]])

    change_status = clean_text(chunk[COL["change_status"]]).str.upper()
    change_date = parse_date_series(chunk[COL["change_date"]])
    prev_fee = to_num(chunk[COL["previous_buyfee"]])
    change_amt = to_num(chunk[COL["change_amount"]])

    unique_locations.update(loc[valid_inventory].dropna().unique().tolist())
    unique_lanes.update(lane[valid_inventory].dropna().unique().tolist())

    sold_with_buyfee += int((sold & buyfee_final.notna()).sum())

    tier_low, tier_high, tier_label = make_price_tier(saleamt_final)

    week_start = (
        sale_dt
        - pd.to_timedelta(sale_dt.dt.weekday, unit="D")
    )

    # Optional vehicle-mix controls
    vehicle_year = (
        to_num(chunk[COL["vehicle_year"]])
        if COL["vehicle_year"]
        else pd.Series(np.nan, index=chunk.index)
    )

    mileage = (
        to_num(chunk[COL["mileage"]])
        if COL["mileage"]
        else pd.Series(np.nan, index=chunk.index)
    )

    high_bid = (
        to_num(chunk[COL["high_bid"]])
        if COL["high_bid"]
        else pd.Series(np.nan, index=chunk.index)
    )

    floor = (
        to_num(chunk[COL["floor"]])
        if COL["floor"]
        else pd.Series(np.nan, index=chunk.index)
    )

    cond_candidates = []

    for k in ["condition_overall", "condition1", "condition2"]:
        if COL[k]:
            cond_candidates.append(to_num(chunk[COL[k]]))

    if cond_candidates:
        condition = (
            pd.concat(cond_candidates, axis=1)
            .bfill(axis=1)
            .iloc[:, 0]
        )
    else:
        condition = pd.Series(np.nan, index=chunk.index)

    make = (
        clean_text(chunk[COL["make"]]).str.upper()
        if COL["make"]
        else pd.Series(pd.NA, index=chunk.index, dtype="string")
    )

    current_year = sale_dt.dt.year
    vehicle_age = current_year - vehicle_year

    # Keep only HST / SLD / NOS inventory
    keep = (
        valid_inventory
        & sale_dt.notna()
        & saleamt_final.notna()
        & (saleamt_final >= 0)
        & loc.notna()
        & lane.notna()
        & tier_label.notna()
    )

    temp = pd.DataFrame({
        "Week_Start": week_start,
        "VRLOCATION": loc,
        "LANE": lane,
        "price_tier": tier_label,
        "price_tier_low": tier_low,
        "price_tier_high": tier_high,

        "Registered_Units": 1,
        "Sold_Units": sold.astype("int8"),
        "NOS_Units": status.eq("NOS").fillna(False).astype("int8"),

        "SaleAmt_Final": saleamt_final,

        # BUYFEE_FINAL only, sold units only
        "BuyFee_Final_Sold": buyfee_final.where(sold, np.nan),
        "Sold_With_BuyFee": (sold & buyfee_final.notna()).astype("int8"),

        "Vehicle_Age": vehicle_age,
        "Mileage": mileage,
        "Condition": condition,
        "High_Bid": high_bid,
        "Floor": floor,
        "Make": make,
    })

    temp = temp.loc[keep].copy()

    weekly_chunk = (
        temp.groupby(
            [
                "Week_Start",
                "VRLOCATION",
                "LANE",
                "price_tier",
                "price_tier_low",
                "price_tier_high"
            ],
            dropna=False,
            sort=False
        )
        .agg(
            Registered_Units=("Registered_Units", "sum"),
            Sold_Units=("Sold_Units", "sum"),
            NOS_Units=("NOS_Units", "sum"),

            SaleAmt_Final_Sum=("SaleAmt_Final", "sum"),
            SaleAmt_Final_Count=("SaleAmt_Final", "count"),

            BuyFee_Revenue=("BuyFee_Final_Sold", "sum"),
            BuyFee_Final_Count=("BuyFee_Final_Sold", "count"),
            Sold_With_BuyFee=("Sold_With_BuyFee", "sum"),

            Vehicle_Age_Sum=("Vehicle_Age", "sum"),
            Vehicle_Age_Count=("Vehicle_Age", "count"),

            Mileage_Sum=("Mileage", "sum"),
            Mileage_Count=("Mileage", "count"),

            Condition_Sum=("Condition", "sum"),
            Condition_Count=("Condition", "count"),

            High_Bid_Sum=("High_Bid", "sum"),
            High_Bid_Count=("High_Bid", "count"),

            Floor_Sum=("Floor", "sum"),
            Floor_Count=("Floor", "count"),

            Unique_Makes=("Make", "nunique"),
        )
        .reset_index()
    )

    weekly_parts.append(weekly_chunk)

    # True fee-change events only
    is_change = change_status.eq("CHANGE").fillna(False)

    event_mask = (
        is_change
        & valid_inventory
        & saleamt_final.notna()
        & loc.notna()
        & lane.notna()
        & change_date.notna()
        & tier_label.notna()
    )

    if event_mask.any():
        ev = pd.DataFrame({
            "VRLOCATION": loc.loc[event_mask],
            "LANE": lane.loc[event_mask],
            "price_tier": tier_label.loc[event_mask],
            "price_tier_low": tier_low.loc[event_mask],
            "price_tier_high": tier_high.loc[event_mask],
            "BUYFEE_CHANGE_DATE": change_date.loc[event_mask],
            "PREVIOUS_BUYFEE": prev_fee.loc[event_mask],
            "BUYFEE_CHANGE_AMOUNT": change_amt.loc[event_mask],
        })

        ev["NEW_BUYFEE"] = (
            ev["PREVIOUS_BUYFEE"]
            + ev["BUYFEE_CHANGE_AMOUNT"]
        )

        event_parts.append(ev)

    if chunk_num % 5 == 0:
        print(
            f"Processed {chunk_num:,} chunks | "
            f"{rows_processed:,} raw rows"
        )

    del chunk, temp, weekly_chunk
    gc.collect()

print("\nPASS 1 COMPLETE")
print("Raw rows processed :", f"{rows_processed:,}")
print("Valid inventory    :", f"{valid_inventory_rows:,}")
print("Sold rows          :", f"{sold_rows:,}")
print("NOS rows           :", f"{nos_rows:,}")
print("Excluded INV       :", f"{excluded_inv_rows:,}")
print("Excluded SAL       :", f"{excluded_sal_rows:,}")
print("Sold w/ BUYFEE_FINAL:", f"{sold_with_buyfee:,}")
print("Unique locations   :", f"{len(unique_locations):,}")
print("Unique lanes       :", f"{len(unique_lanes):,}")


Processed 5 chunks | 1,250,000 raw rows
Processed 10 chunks | 2,500,000 raw rows
Processed 15 chunks | 3,750,000 raw rows
Processed 20 chunks | 5,000,000 raw rows

PASS 1 COMPLETE
Raw rows processed : 5,540,598
Valid inventory    : 5,528,661
Sold rows          : 149,463
NOS rows           : 2,265,049
Excluded INV       : 3,389
Excluded SAL       : 8,548
Sold w/ BUYFEE_FINAL: 55,269
Unique locations   : 54
Unique lanes       : 361


## 4. Explicit status and unique-location diagnostics

In [5]:

status_diag = (
    pd.DataFrame(
        [{"VRSTATUS": k, "Count": v} for k, v in status_counts.items()]
    )
    .sort_values("Count", ascending=False)
)

status_diag.to_csv(
    OUTPUT_DIR / "01_STATUS_DIAGNOSTICS.csv",
    index=False
)

location_diag = pd.DataFrame({
    "Metric": [
        "Unique_Valid_Locations",
        "Unique_Valid_Lanes",
        "Raw_Rows_Processed",
        "Valid_Inventory_Rows",
        "Sold_Rows",
        "NOS_Rows",
        "Excluded_INV_Rows",
        "Excluded_SAL_Rows",
        "Sold_With_BUYFEE_FINAL",
        "Sold_BUYFEE_FINAL_Coverage"
    ],
    "Value": [
        len(unique_locations),
        len(unique_lanes),
        rows_processed,
        valid_inventory_rows,
        sold_rows,
        nos_rows,
        excluded_inv_rows,
        excluded_sal_rows,
        sold_with_buyfee,
        sold_with_buyfee / sold_rows if sold_rows else np.nan
    ]
})

location_diag.to_csv(
    OUTPUT_DIR / "02_CORE_DIAGNOSTICS.csv",
    index=False
)

print("Status counts:")
display(status_diag)

print("\nCore diagnostics:")
display(location_diag)

print("\nUnique valid locations:")
print(sorted(unique_locations))


Status counts:


,VRSTATUS,Count
0,HST,3263450
1,NOS,2265049
2,SAL,8548
3,INV,3389
4,SLD,162



Core diagnostics:


,Metric,Value
0,Unique_Valid_Locations,5.400000e+01
1,Unique_Valid_Lanes,3.610000e+02
2,Raw_Rows_Processed,5.540598e+06
3,Valid_Inventory_Rows,5.528661e+06
4,Sold_Rows,1.494630e+05
5,NOS_Rows,2.265049e+06
6,Excluded_INV_Rows,3.389000e+03
7,Excluded_SAL_Rows,8.548000e+03
8,Sold_With_BUYFEE_FINAL,5.526900e+04
9,Sold_BUYFEE_FINAL_Coverage,3.697838e-01



Unique valid locations:
['ALBANY', 'ATLANTA', 'AUCTIONINMOTION', 'AUGUSTA', 'AUSTIN', 'BADGERSTATE', 'BATONROUGE', 'BIRMINGHAM', 'BOSTON', 'BOWLINGGREEN', 'CENTRALHOUSTON', 'CHARLESTON', 'CHICAGO', 'CHIPPEWAFALLS', 'CLARKCOUNTY', 'COLUMBUS', 'CORPUSCHRISTI', 'CORRY', 'DALLAS', 'DENVER', 'ELPASO', 'GREENVILLE', 'HARRISBURG', 'HOUSTON', 'IOWA', 'JACKSONVILLE', 'KALAMAZOO', 'KANSAS', 'LAKECHARLES', 'LANCASTER', 'LASVEGAS', 'LUBBOCK', 'MINNESOTA', 'MORTON', 'NEWJERSEY', 'NEWORLEANS', 'NEWYORK', 'NORTHCALIFORNIA', 'NORTHHOUSTON', 'PENSACOLA', 'PHOENIX', 'PITTSBURGH', 'RICHMOND', 'SAINTLOUIS', 'SANANTONIO', 'SAVANNAH', 'SHREVEPORT', 'SOUTHWISCONSIN', 'TAMPA', 'TOLEDO', 'TULSA', 'VICTORIA', 'VIRGINIA', 'WESTMICHIGAN']


## 5. Consolidate weekly model panel

In [6]:

weekly_raw = pd.concat(weekly_parts, ignore_index=True)

group_cols = [
    "Week_Start",
    "VRLOCATION",
    "LANE",
    "price_tier",
    "price_tier_low",
    "price_tier_high"
]

sum_cols = [c for c in weekly_raw.columns if c not in group_cols]

weekly = (
    weekly_raw.groupby(
        group_cols,
        dropna=False,
        sort=False
    )[sum_cols]
    .sum(min_count=1)
    .reset_index()
)

weekly["Conversion"] = np.where(
    weekly["Registered_Units"] > 0,
    weekly["Sold_Units"] / weekly["Registered_Units"],
    np.nan
)

weekly["Avg_SaleAmt_Final"] = np.where(
    weekly["SaleAmt_Final_Count"] > 0,
    weekly["SaleAmt_Final_Sum"] / weekly["SaleAmt_Final_Count"],
    np.nan
)

weekly["BuyFee_RPU"] = np.where(
    weekly["Sold_Units"] > 0,
    weekly["BuyFee_Revenue"] / weekly["Sold_Units"],
    np.nan
)

weekly["BuyFee_Coverage"] = np.where(
    weekly["Sold_Units"] > 0,
    weekly["Sold_With_BuyFee"] / weekly["Sold_Units"],
    np.nan
)

weekly["Avg_BuyFee_Final"] = np.where(
    weekly["BuyFee_Final_Count"] > 0,
    weekly["BuyFee_Revenue"] / weekly["BuyFee_Final_Count"],
    np.nan
)

weekly["Avg_Vehicle_Age"] = np.where(
    weekly["Vehicle_Age_Count"] > 0,
    weekly["Vehicle_Age_Sum"] / weekly["Vehicle_Age_Count"],
    np.nan
)

weekly["Avg_Mileage"] = np.where(
    weekly["Mileage_Count"] > 0,
    weekly["Mileage_Sum"] / weekly["Mileage_Count"],
    np.nan
)

weekly["Avg_Condition"] = np.where(
    weekly["Condition_Count"] > 0,
    weekly["Condition_Sum"] / weekly["Condition_Count"],
    np.nan
)

weekly["Avg_High_Bid"] = np.where(
    weekly["High_Bid_Count"] > 0,
    weekly["High_Bid_Sum"] / weekly["High_Bid_Count"],
    np.nan
)

weekly["Avg_Floor"] = np.where(
    weekly["Floor_Count"] > 0,
    weekly["Floor_Sum"] / weekly["Floor_Count"],
    np.nan
)

weekly["Year"] = weekly["Week_Start"].dt.year
weekly["Month"] = weekly["Week_Start"].dt.month
weekly["Quarter"] = weekly["Week_Start"].dt.quarter
weekly["Week_of_Year"] = weekly["Week_Start"].dt.isocalendar().week.astype(int)

weekly = weekly.sort_values(
    ["VRLOCATION", "LANE", "price_tier_low", "Week_Start"]
).reset_index(drop=True)

weekly.to_csv(
    OUTPUT_DIR / "03_WEEKLY_MODEL_PANEL_V6.csv",
    index=False
)

print("Weekly panel rows:", f"{len(weekly):,}")
print("Weekly unique locations:", weekly["VRLOCATION"].nunique())
print("Median registered/week:", weekly["Registered_Units"].median())
print("Median sold/week:", weekly["Sold_Units"].median())
print("Weekly RPU non-null %:", round(weekly["BuyFee_RPU"].notna().mean() * 100, 2))
print("Weekly fee coverage median:", weekly["BuyFee_Coverage"].median())


Weekly panel rows: 1,689,856
Weekly unique locations: 54
Median registered/week: 2.0
Median sold/week: 0.0
Weekly RPU non-null %: 3.87
Weekly fee coverage median: 0.0


## 6. Build clean fee-change table with hierarchical IDs

In [7]:

events_raw = pd.concat(event_parts, ignore_index=True)

event_key = [
    "VRLOCATION",
    "LANE",
    "price_tier",
    "BUYFEE_CHANGE_DATE"
]

events = (
    events_raw.groupby(
        event_key,
        as_index=False,
        dropna=False
    )
    .agg(
        price_tier_low=("price_tier_low", "first"),
        price_tier_high=("price_tier_high", "first"),
        PREVIOUS_BUYFEE=("PREVIOUS_BUYFEE", "median"),
        BUYFEE_CHANGE_AMOUNT=("BUYFEE_CHANGE_AMOUNT", "median"),
        NEW_BUYFEE=("NEW_BUYFEE", "median"),
        Source_Rows=("NEW_BUYFEE", "size"),
    )
)

events["BUYFEE_CHANGE_DATE"] = pd.to_datetime(
    events["BUYFEE_CHANGE_DATE"],
    errors="coerce"
)

events["FEE_CHANGE_PCT"] = np.where(
    events["PREVIOUS_BUYFEE"].abs() > 0,
    events["BUYFEE_CHANGE_AMOUNT"] / events["PREVIOUS_BUYFEE"],
    np.nan
)

events["FEE_DIRECTION"] = np.select(
    [
        events["BUYFEE_CHANGE_AMOUNT"] > 0,
        events["BUYFEE_CHANGE_AMOUNT"] < 0
    ],
    ["Increase", "Decrease"],
    default="No Change"
)

events["Corporate_Change_ID"] = (
    events["BUYFEE_CHANGE_DATE"].dt.strftime("%Y-%m-%d")
)

events["Location_Change_ID"] = (
    events["VRLOCATION"].astype(str)
    + "_"
    + events["Corporate_Change_ID"]
)

events["Segment_Change_ID"] = (
    events["VRLOCATION"].astype(str)
    + "_"
    + events["LANE"].astype(str)
    + "_"
    + events["price_tier"].astype(str)
    + "_"
    + events["Corporate_Change_ID"]
)

events = events.sort_values(
    ["VRLOCATION", "LANE", "price_tier", "BUYFEE_CHANGE_DATE"]
).reset_index(drop=True)

# Overlap flags
events["Prev_Change_Date"] = (
    events.groupby(
        ["VRLOCATION", "LANE", "price_tier"]
    )["BUYFEE_CHANGE_DATE"].shift(1)
)

events["Next_Change_Date"] = (
    events.groupby(
        ["VRLOCATION", "LANE", "price_tier"]
    )["BUYFEE_CHANGE_DATE"].shift(-1)
)

events["Weeks_Since_Prior_Change"] = (
    (events["BUYFEE_CHANGE_DATE"] - events["Prev_Change_Date"]).dt.days / 7
)

events["Weeks_To_Next_Change"] = (
    (events["Next_Change_Date"] - events["BUYFEE_CHANGE_DATE"]).dt.days / 7
)

events["Overlap_Flag"] = (
    events["Weeks_Since_Prior_Change"].lt(OVERLAP_LOOKBACK_WEEKS)
    |
    events["Weeks_To_Next_Change"].lt(OVERLAP_LOOKAHEAD_WEEKS)
).fillna(False)

events.to_csv(
    OUTPUT_DIR / "04_CLEAN_FEE_CHANGE_EVENTS_V6.csv",
    index=False
)

print("Segment fee events:", f"{len(events):,}")
print("Unique locations:", events["VRLOCATION"].nunique())
print(
    "Unique Location × Change Date:",
    events[["VRLOCATION", "BUYFEE_CHANGE_DATE"]]
    .drop_duplicates()
    .shape[0]
)
print("Unique corporate change dates:", events["BUYFEE_CHANGE_DATE"].nunique())
print("Increase events:", int((events["FEE_DIRECTION"] == "Increase").sum()))
print("Decrease events:", int((events["FEE_DIRECTION"] == "Decrease").sum()))


Segment fee events: 9,251
Unique locations: 38
Unique Location × Change Date: 103
Unique corporate change dates: 51
Increase events: 9009
Decrease events: 242


## 7. Build 4W / 8W / 12W event-level dataset

In [8]:

panel_lookup = {
    key: grp.sort_values("Week_Start").copy()
    for key, grp in weekly.groupby(
        ["VRLOCATION", "LANE", "price_tier"],
        sort=False
    )
}

def summarize_period(df):
    if df.empty:
        return {
            "observed_weeks": 0,
            "registered": 0,
            "sold": 0,
            "conversion": np.nan,
            "buyfee_revenue": 0.0,
            "buyfee_rpu": np.nan,
            "buyfee_coverage": np.nan,
            "avg_saleamt": np.nan,
            "avg_age": np.nan,
            "avg_mileage": np.nan,
            "avg_condition": np.nan,
            "avg_high_bid": np.nan,
            "avg_floor": np.nan,
            "volume_trend": np.nan,
            "conversion_trend": np.nan,
            "volume_cv": np.nan,
            "conversion_cv": np.nan,
        }

    reg = df["Registered_Units"].sum()
    sold = df["Sold_Units"].sum()

    fee_rev = df["BuyFee_Revenue"].sum()
    sold_with_fee = df["Sold_With_BuyFee"].sum()

    def wmean(sum_col, count_col):
        den = df[count_col].sum()
        if den <= 0:
            return np.nan
        return df[sum_col].sum() / den

    return {
        "observed_weeks": df["Week_Start"].nunique(),
        "registered": reg,
        "sold": sold,
        "conversion": sold / reg if reg > 0 else np.nan,
        "buyfee_revenue": fee_rev,
        "buyfee_rpu": fee_rev / sold if sold > 0 else np.nan,
        "buyfee_coverage": sold_with_fee / sold if sold > 0 else np.nan,
        "avg_saleamt": wmean("SaleAmt_Final_Sum", "SaleAmt_Final_Count"),
        "avg_age": wmean("Vehicle_Age_Sum", "Vehicle_Age_Count"),
        "avg_mileage": wmean("Mileage_Sum", "Mileage_Count"),
        "avg_condition": wmean("Condition_Sum", "Condition_Count"),
        "avg_high_bid": wmean("High_Bid_Sum", "High_Bid_Count"),
        "avg_floor": wmean("Floor_Sum", "Floor_Count"),
        "volume_trend": linear_slope(df["Sold_Units"].values),
        "conversion_trend": linear_slope(df["Conversion"].values),
        "volume_cv": coefficient_of_variation(df["Sold_Units"].values),
        "conversion_cv": coefficient_of_variation(df["Conversion"].values),
    }

rows = []

for i, e in events.iterrows():
    key = (
        e["VRLOCATION"],
        e["LANE"],
        e["price_tier"]
    )

    seg = panel_lookup.get(key)

    if seg is None or seg.empty:
        continue

    event_date = pd.Timestamp(e["BUYFEE_CHANGE_DATE"])

    event_week = (
        event_date
        - pd.to_timedelta(event_date.weekday(), unit="D")
    )

    out = e.to_dict()
    out["Event_Week"] = event_week

    for w in WINDOWS:
        pre_start = event_week - pd.Timedelta(weeks=w)
        pre_end = event_week - pd.Timedelta(weeks=1)

        post_start = event_week + pd.Timedelta(weeks=1)
        post_end = event_week + pd.Timedelta(weeks=w)

        pre = seg[
            seg["Week_Start"].between(pre_start, pre_end)
        ]

        post = seg[
            seg["Week_Start"].between(post_start, post_end)
        ]

        a = summarize_period(pre)
        b = summarize_period(post)

        for k, v in a.items():
            out[f"pre_{w}w_{k}"] = v

        for k, v in b.items():
            out[f"post_{w}w_{k}"] = v

        out[f"volume_change_pct_{w}w"] = safe_pct_change(
            b["sold"], a["sold"]
        )

        out[f"registered_change_pct_{w}w"] = safe_pct_change(
            b["registered"], a["registered"]
        )

        out[f"conversion_change_pp_{w}w"] = (
            (b["conversion"] - a["conversion"]) * 100
            if pd.notna(a["conversion"]) and pd.notna(b["conversion"])
            else np.nan
        )

        out[f"rpu_dollar_change_{w}w"] = (
            b["buyfee_rpu"] - a["buyfee_rpu"]
            if pd.notna(a["buyfee_rpu"]) and pd.notna(b["buyfee_rpu"])
            else np.nan
        )

        out[f"fee_pass_through_{w}w"] = (
            out[f"rpu_dollar_change_{w}w"] / e["BUYFEE_CHANGE_AMOUNT"]
            if pd.notna(out[f"rpu_dollar_change_{w}w"])
            and pd.notna(e["BUYFEE_CHANGE_AMOUNT"])
            and e["BUYFEE_CHANGE_AMOUNT"] != 0
            else np.nan
        )

    rows.append(out)

    if (i + 1) % 1000 == 0:
        print(f"Processed {i+1:,} events")

event_model = pd.DataFrame(rows)

event_model.to_csv(
    OUTPUT_DIR / "05_EVENT_LEVEL_MODEL_DATA_RAW_V6.csv",
    index=False
)

print("Raw event-model rows:", f"{len(event_model):,}")


Processed 1,000 events
Processed 2,000 events
Processed 3,000 events
Processed 4,000 events
Processed 5,000 events
Processed 6,000 events
Processed 7,000 events
Processed 8,000 events
Processed 9,000 events
Raw event-model rows: 9,251


## 8. Apply exact V6 eligibility rules

In [9]:

w = PRIMARY_WINDOW

# ------------------------------------------------------------
# 6-of-8 observed-week checks
# ------------------------------------------------------------

event_model["Observed_Week_Eligible"] = (
    (event_model[f"pre_{w}w_observed_weeks"] >= MIN_OBSERVED_WEEKS_PRE)
    &
    (event_model[f"post_{w}w_observed_weeks"] >= MIN_OBSERVED_WEEKS_POST)
)

# ------------------------------------------------------------
# >=25 registered pre and post
# ------------------------------------------------------------

event_model["Registered_Exposure_Eligible"] = (
    (event_model[f"pre_{w}w_registered"] >= MIN_REGISTERED_PRE)
    &
    (event_model[f"post_{w}w_registered"] >= MIN_REGISTERED_POST)
)

# ------------------------------------------------------------
# Volume / conversion eligibility
# ------------------------------------------------------------

event_model["Volume_Eligible"] = (
    event_model["Observed_Week_Eligible"]
    &
    event_model["Registered_Exposure_Eligible"]
    &
    (event_model[f"pre_{w}w_sold"] > 0)
    &
    event_model[f"volume_change_pct_{w}w"].notna()
)

event_model["Conversion_Eligible"] = (
    event_model["Observed_Week_Eligible"]
    &
    event_model["Registered_Exposure_Eligible"]
    &
    event_model[f"conversion_change_pp_{w}w"].notna()
)

# ------------------------------------------------------------
# RPU eligibility
# ------------------------------------------------------------

event_model["RPU_Eligible"] = (
    event_model["Observed_Week_Eligible"]
    &
    event_model["Registered_Exposure_Eligible"]
    &
    (event_model[f"pre_{w}w_sold"] >= MIN_SOLD_PRE_RPU)
    &
    (event_model[f"post_{w}w_sold"] >= MIN_SOLD_POST_RPU)
    &
    (event_model[f"pre_{w}w_buyfee_coverage"] >= MIN_BUYFEE_COVERAGE)
    &
    (event_model[f"post_{w}w_buyfee_coverage"] >= MIN_BUYFEE_COVERAGE)
    &
    event_model[f"rpu_dollar_change_{w}w"].notna()
)

event_model["Primary_Model_Eligible"] = (
    event_model["Volume_Eligible"]
    &
    event_model["Conversion_Eligible"]
)

event_model["Strict_Model_Eligible"] = (
    event_model["Primary_Model_Eligible"]
    &
    ~event_model["Overlap_Flag"]
)

event_model.to_csv(
    OUTPUT_DIR / "06_EVENT_LEVEL_MODEL_DATA_WITH_FLAGS_V6.csv",
    index=False
)

primary = event_model[
    event_model["Primary_Model_Eligible"]
].copy()

strict = event_model[
    event_model["Strict_Model_Eligible"]
].copy()

rpu_ready = event_model[
    event_model["RPU_Eligible"]
].copy()

primary.to_csv(
    OUTPUT_DIR / "07_MODEL_READY_PRIMARY_V6.csv",
    index=False
)

strict.to_csv(
    OUTPUT_DIR / "08_MODEL_READY_STRICT_NO_OVERLAP_V6.csv",
    index=False
)

rpu_ready.to_csv(
    OUTPUT_DIR / "09_MODEL_READY_RPU_V6.csv",
    index=False
)

print("Observed-week eligible:", f"{event_model['Observed_Week_Eligible'].sum():,}")
print("Registered exposure eligible:", f"{event_model['Registered_Exposure_Eligible'].sum():,}")
print("Volume eligible:", f"{event_model['Volume_Eligible'].sum():,}")
print("Conversion eligible:", f"{event_model['Conversion_Eligible'].sum():,}")
print("RPU eligible:", f"{event_model['RPU_Eligible'].sum():,}")
print("Primary model eligible:", f"{len(primary):,}")
print("Strict no-overlap eligible:", f"{len(strict):,}")


Observed-week eligible: 4,062
Registered exposure eligible: 2,221
Volume eligible: 84
Conversion eligible: 2,142
RPU eligible: 0
Primary model eligible: 84
Strict no-overlap eligible: 84


## 9. Explicit unique-location and event diagnostics

In [10]:

diagnostic_rows = []

def add(metric, value):
    diagnostic_rows.append({
        "Metric": metric,
        "Value": value
    })

add("Unique_Raw_Valid_Locations", len(unique_locations))
add("Weekly_Panel_Unique_Locations", weekly["VRLOCATION"].nunique())
add("Fee_Event_Unique_Locations", events["VRLOCATION"].nunique())
add("Primary_Model_Unique_Locations", primary["VRLOCATION"].nunique())
add("Strict_Model_Unique_Locations", strict["VRLOCATION"].nunique())
add("RPU_Model_Unique_Locations", rpu_ready["VRLOCATION"].nunique())

add("Segment_Fee_Change_Events", len(events))
add(
    "Location_x_Change_Date_Events",
    events[["VRLOCATION", "BUYFEE_CHANGE_DATE"]]
    .drop_duplicates()
    .shape[0]
)
add(
    "Corporate_Change_Dates",
    events["BUYFEE_CHANGE_DATE"].nunique()
)

add("Primary_Model_Events", len(primary))
add("Strict_No_Overlap_Events", len(strict))
add("RPU_Eligible_Events", len(rpu_ready))

add(
    "Sold_BUYFEE_FINAL_Coverage_Overall",
    sold_with_buyfee / sold_rows if sold_rows else np.nan
)

diagnostics = pd.DataFrame(diagnostic_rows)

diagnostics.to_csv(
    OUTPUT_DIR / "10_UNIQUE_LOCATION_AND_EVENT_DIAGNOSTICS.csv",
    index=False
)

display(diagnostics)


,Metric,Value
0,Unique_Raw_Valid_Locations,54.000000
1,Weekly_Panel_Unique_Locations,54.000000
2,Fee_Event_Unique_Locations,38.000000
3,Primary_Model_Unique_Locations,2.000000
4,Strict_Model_Unique_Locations,2.000000
5,RPU_Model_Unique_Locations,0.000000
6,Segment_Fee_Change_Events,9251.000000
7,Location_x_Change_Date_Events,103.000000
8,Corporate_Change_Dates,51.000000
9,Primary_Model_Events,84.000000


## 10. Threshold and 6-of-8 sensitivity

In [11]:

rows = []

for reg_threshold in [10, 25, 50, 100]:
    for week_threshold in [4, 6, 8]:
        eligible = (
            (event_model[f"pre_{PRIMARY_WINDOW}w_registered"] >= reg_threshold)
            &
            (event_model[f"post_{PRIMARY_WINDOW}w_registered"] >= reg_threshold)
            &
            (event_model[f"pre_{PRIMARY_WINDOW}w_observed_weeks"] >= week_threshold)
            &
            (event_model[f"post_{PRIMARY_WINDOW}w_observed_weeks"] >= week_threshold)
        )

        rows.append({
            "Registered_Threshold": reg_threshold,
            "Observed_Weeks_Threshold": week_threshold,
            "Eligible_Events": int(eligible.sum()),
            "Eligible_Pct": float(eligible.mean())
        })

threshold_sensitivity = pd.DataFrame(rows)

threshold_sensitivity.to_csv(
    OUTPUT_DIR / "11_THRESHOLD_AND_WEEK_SENSITIVITY.csv",
    index=False
)

display(threshold_sensitivity)


,Registered_Threshold,Observed_Weeks_Threshold,Eligible_Events,Eligible_Pct
0,10,4,4450,0.481029
1,10,6,3884,0.419847
2,10,8,1688,0.182467
3,25,4,2202,0.238028
4,25,6,2142,0.231543
5,25,8,1425,0.154037
6,50,4,907,0.098043
7,50,6,894,0.096638
8,50,8,693,0.074911
9,100,4,178,0.019241


## 11. Historical support for +$10 / +$15 / +$20 / +$25 / +$30

In [12]:

scenario_rows = []

for target in [10, 15, 20, 25, 30]:
    support = events[
        events["BUYFEE_CHANGE_AMOUNT"].between(
            target - 2.5,
            target + 2.5
        )
    ]

    primary_support = primary[
        primary["BUYFEE_CHANGE_AMOUNT"].between(
            target - 2.5,
            target + 2.5
        )
    ]

    scenario_rows.append({
        "Scenario_Fee_Increase": target,
        "All_Segment_Events": len(support),
        "All_Unique_Locations": support["VRLOCATION"].nunique(),
        "All_Location_Change_Events": (
            support[["VRLOCATION", "BUYFEE_CHANGE_DATE"]]
            .drop_duplicates()
            .shape[0]
        ),
        "All_Corporate_Change_Dates": support["BUYFEE_CHANGE_DATE"].nunique(),
        "Primary_Model_Eligible_Events": len(primary_support),
        "Primary_Model_Unique_Locations": primary_support["VRLOCATION"].nunique(),
        "Primary_Model_Corporate_Change_Dates": primary_support["BUYFEE_CHANGE_DATE"].nunique(),
    })

scenario_support = pd.DataFrame(scenario_rows)

scenario_support.to_csv(
    OUTPUT_DIR / "12_SCENARIO_HISTORICAL_SUPPORT_V6.csv",
    index=False
)

display(scenario_support)


,Scenario_Fee_Increase,All_Segment_Events,All_Unique_Locations,All_Location_Change_Events,All_Corporate_Change_Dates,Primary_Model_Eligible_Events,Primary_Model_Unique_Locations,Primary_Model_Corporate_Change_Dates
0,10,2600,18,48,31,4,2,2
1,15,329,17,24,16,2,1,1
2,20,1034,17,31,19,1,1,1
3,25,467,18,22,13,0,0,0
4,30,323,17,25,17,0,0,0


## 12. Final validation summary

In [13]:

summary_rows = []

def add_summary(metric, value):
    summary_rows.append({
        "Metric": metric,
        "Value": value
    })

add_summary("Raw_Rows_Processed", rows_processed)
add_summary("Valid_HST_SLD_NOS_Rows", valid_inventory_rows)
add_summary("Sold_Rows_Exact_Logic", sold_rows)
add_summary("NOS_Rows", nos_rows)
add_summary("Excluded_INV_Rows", excluded_inv_rows)
add_summary("Excluded_SAL_Rows", excluded_sal_rows)

add_summary("Unique_Valid_Locations", len(unique_locations))
add_summary("Unique_Valid_Lanes", len(unique_lanes))

add_summary("Sold_With_BUYFEE_FINAL", sold_with_buyfee)
add_summary(
    "Sold_BUYFEE_FINAL_Coverage",
    sold_with_buyfee / sold_rows if sold_rows else np.nan
)

add_summary("Weekly_Panel_Rows", len(weekly))
add_summary("Weekly_Panel_Unique_Locations", weekly["VRLOCATION"].nunique())
add_summary("Median_Weekly_Registered", weekly["Registered_Units"].median())
add_summary("Median_Weekly_Sold", weekly["Sold_Units"].median())

add_summary("Segment_Fee_Change_Events", len(events))
add_summary("Fee_Event_Unique_Locations", events["VRLOCATION"].nunique())
add_summary(
    "Location_x_Change_Date_Events",
    events[["VRLOCATION", "BUYFEE_CHANGE_DATE"]]
    .drop_duplicates()
    .shape[0]
)
add_summary("Corporate_Change_Dates", events["BUYFEE_CHANGE_DATE"].nunique())

add_summary("Observed_Week_Eligible", int(event_model["Observed_Week_Eligible"].sum()))
add_summary("Registered_Exposure_Eligible", int(event_model["Registered_Exposure_Eligible"].sum()))
add_summary("Volume_Eligible", int(event_model["Volume_Eligible"].sum()))
add_summary("Conversion_Eligible", int(event_model["Conversion_Eligible"].sum()))
add_summary("RPU_Eligible", int(event_model["RPU_Eligible"].sum()))
add_summary("Primary_Model_Eligible", len(primary))
add_summary("Strict_No_Overlap_Eligible", len(strict))

validation_summary = pd.DataFrame(summary_rows)

validation_summary.to_csv(
    OUTPUT_DIR / "13_VALIDATION_SUMMARY_V6.csv",
    index=False
)

display(validation_summary)


,Metric,Value
0,Raw_Rows_Processed,5.540598e+06
1,Valid_HST_SLD_NOS_Rows,5.528661e+06
2,Sold_Rows_Exact_Logic,1.494630e+05
3,NOS_Rows,2.265049e+06
4,Excluded_INV_Rows,3.389000e+03
5,Excluded_SAL_Rows,8.548000e+03
6,Unique_Valid_Locations,5.400000e+01
7,Unique_Valid_Lanes,3.610000e+02
8,Sold_With_BUYFEE_FINAL,5.526900e+04
9,Sold_BUYFEE_FINAL_Coverage,3.697838e-01


## 13. QA checklist

In [14]:

qa = pd.DataFrame([
    {
        "Check": "Exact sold logic",
        "Status": "PASS",
        "Detail": '(HST and VRINVOICE != RETURN) OR (SLD and VRIFORTRY != Y)'
    },
    {
        "Check": "NOS treatment",
        "Status": "PASS",
        "Detail": "NOS retained as valid unsold inventory."
    },
    {
        "Check": "INV exclusion",
        "Status": "PASS",
        "Detail": "INV excluded from standard modeling inventory."
    },
    {
        "Check": "SAL exclusion",
        "Status": "PASS",
        "Detail": "SAL excluded from standard modeling inventory."
    },
    {
        "Check": "Price tier source",
        "Status": "PASS",
        "Detail": "SALEAMT_FINAL used for sold and unsold units."
    },
    {
        "Check": "Buy Fee source",
        "Status": "PASS",
        "Detail": "BUYFEE_FINAL only. Buyer_BuyFee is not used."
    },
    {
        "Check": "Fee-change treatment",
        "Status": "PASS",
        "Detail": 'Only BUYFEE_CHANGE_STATUS == "Change" creates an event.'
    },
    {
        "Check": "6-of-8 observed-week rule",
        "Status": "PASS",
        "Detail": "Primary 8W dataset requires >=6 observed weeks pre and post."
    },
    {
        "Check": "25 registered-unit rule",
        "Status": "PASS",
        "Detail": "Primary 8W dataset requires >=25 registered units pre and post."
    },
    {
        "Check": "Unique-location diagnostics",
        "Status": "PASS",
        "Detail": "Unique locations are reported separately from Location × Change Date events."
    },
])

qa.to_csv(
    OUTPUT_DIR / "14_QA_CHECKLIST_V6.csv",
    index=False
)

display(qa)


,Check,Status,Detail
0,Exact sold logic,PASS,(HST and VRINVOICE != RETURN) OR (SLD and VRIF...
1,NOS treatment,PASS,NOS retained as valid unsold inventory.
2,INV exclusion,PASS,INV excluded from standard modeling inventory.
3,SAL exclusion,PASS,SAL excluded from standard modeling inventory.
4,Price tier source,PASS,SALEAMT_FINAL used for sold and unsold units.
5,Buy Fee source,PASS,BUYFEE_FINAL only. Buyer_BuyFee is not used.
6,Fee-change treatment,PASS,"Only BUYFEE_CHANGE_STATUS == ""Change"" creates ..."
7,6-of-8 observed-week rule,PASS,Primary 8W dataset requires >=6 observed weeks...
8,25 registered-unit rule,PASS,Primary 8W dataset requires >=25 registered un...
9,Unique-location diagnostics,PASS,Unique locations are reported separately from ...



## Files to share back

After the notebook runs, please upload these:

1. `13_VALIDATION_SUMMARY_V6.csv`
2. `10_UNIQUE_LOCATION_AND_EVENT_DIAGNOSTICS.csv`
3. `11_THRESHOLD_AND_WEEK_SENSITIVITY.csv`
4. `12_SCENARIO_HISTORICAL_SUPPORT_V6.csv`
5. `14_QA_CHECKLIST_V6.csv`
6. `04_CLEAN_FEE_CHANGE_EVENTS_V6.csv`
7. `07_MODEL_READY_PRIMARY_V6.csv`
8. `09_MODEL_READY_RPU_V6.csv`

If the model-ready files are large, the first five plus a sample of the last two is enough for review.

## What we will decide next

Once these outputs are reviewed, we can decide whether:

- the core dataset is now trustworthy,
- the 6-of-8 and 25-unit thresholds are appropriate,
- $1,000 tiers remain too sparse,
- BUYFEE_FINAL coverage is sufficient for RPU modeling,
- overlapping fee changes need to be excluded,
- and whether we are ready to proceed to the causal + ML training stage.
